# 📊 Comparación entre RDD y DataFrame en PySpark

En este notebook se exploran las diferencias y similitudes entre el uso de **RDDs** y **DataFrames** en PySpark, mediante ejemplos prácticos que muestran operaciones de transformación, paralelización y procesamiento distribuido sobre un conjunto de datos bancarios.

## ¿Qué es un RDD?

Un **RDD** (*Resilient Distributed Dataset*) es la estructura de datos fundamental de Apache Spark. Se trata de una colección distribuida e inmutable de objetos que puede procesarse en paralelo a través de múltiples nodos. Los RDDs ofrecen un control fino sobre las operaciones y la ejecución, aunque con una sintaxis más detallada y menos optimizada que la de los DataFrames.

Por el contrario, los **DataFrames** proporcionan una abstracción de más alto nivel, similar a una tabla SQL, lo que permite escribir código más limpio, legible y con mejor rendimiento gracias al motor de optimización de Spark.



In [31]:
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
from pyspark.sql import SparkSession
from pyspark.sql.functions import avg, max as spark_max, min as spark_min

In [32]:
# Configuración del entorno de Spark
spark = SparkSession.builder.appName("AnalisisExploratorio").getOrCreate()

### Comparar operaciones RDD vs DataFrame

In [33]:
df = spark.read.csv("./data/Bank_Customer_Churn_Prediction.csv", header=True, inferSchema=True)
df.show(5)
df.printSchema()

+-----------+------------+-------+------+---+------+---------+---------------+-----------+-------------+----------------+-----+
|customer_id|credit_score|country|gender|age|tenure|  balance|products_number|credit_card|active_member|estimated_salary|churn|
+-----------+------------+-------+------+---+------+---------+---------------+-----------+-------------+----------------+-----+
|   15634602|         619| France|Female| 42|     2|      0.0|              1|          1|            1|       101348.88|    1|
|   15647311|         608|  Spain|Female| 41|     1| 83807.86|              1|          0|            1|       112542.58|    0|
|   15619304|         502| France|Female| 42|     8| 159660.8|              3|          1|            0|       113931.57|    1|
|   15701354|         699| France|Female| 39|     1|      0.0|              2|          0|            0|        93826.63|    0|
|   15737888|         850|  Spain|Female| 43|     2|125510.82|              1|          1|            1|

#### Comparación de operaciones utilizando DataFrames y RDD

Calculamos el saldo promedio de los clientes utilizando tanto RDD como DataFrame para comparar la sintaxis y el resultado.

In [34]:
# Usando DataFrame
promedio_df = df.agg(avg("balance")).collect()[0][0]
print("Promedio (DataFrame):", promedio_df)

# Usando RDD
balances = df.select("balance").rdd.flatMap(lambda x: x)
promedio_rdd = balances.mean()
print("Promedio (RDD):", promedio_rdd)

Promedio (DataFrame): 76485.88928799961
Promedio (RDD): 76485.88928800033


Calcular saldo promedio por pais

In [41]:
# Usando DataFrame
df.groupBy("country").agg(avg("balance")).show()

# Usando RDD
# (país, saldo)
paises_saldos = df.select("country", "balance").rdd.map(lambda row: (row["country"], row["balance"]))
# (país, (suma_saldos, cantidad))
suma_cant = paises_saldos.aggregateByKey((0.0, 0), 
    lambda acc, v: (acc[0] + v, acc[1] + 1),       # función de agregación dentro de cada partición
    lambda acc1, acc2: (acc1[0] + acc2[0], acc1[1] + acc2[1]) #función para combinar los resultados de particiones distintas
)
promedios = suma_cant.mapValues(lambda x: x[0]/x[1] if x[1] > 0 else None).collect()
promedios

+-------+------------------+
|country|      avg(balance)|
+-------+------------------+
|Germany|119730.11613391782|
| France|  62092.6365157559|
|  Spain| 61818.14776342349|
+-------+------------------+



[('France', 62092.6365157559),
 ('Spain', 61818.14776342349),
 ('Germany', 119730.11613391782)]

Calcular el promedio de credit_score por género y país

In [36]:
from pyspark.sql.functions import avg

# Usando DataFrame 
df.groupBy("country", "gender") \
  .agg(avg("credit_score").alias("avg_credit_score")) \
  .show()
  

# Usando RDD
# Creamos un RDD con clave (country, gender) y valor credit_score
rdd = df.select("country", "gender", "credit_score").rdd
pair_rdd = rdd.map(lambda row: ((row["country"], row["gender"]), row["credit_score"]))
# Usamos aggregateByKey para calcular (suma, cuenta)
sum_count_rdd = pair_rdd.aggregateByKey((0, 0),
    lambda acc, val: (acc[0] + val, acc[1] + 1),
    lambda a, b: (a[0] + b[0], a[1] + b[1])
)
# Calculamos el promedio
avg_credit_score_rdd = sum_count_rdd.mapValues(lambda x: x[0] / x[1] if x[1] > 0 else None)
avg_credit_score_rdd.collect()

+-------+------+-----------------+
|country|gender| avg_credit_score|
+-------+------+-----------------+
|Germany|Female|653.0938809723386|
| France|  Male|650.0646567381038|
| France|Female|649.1857585139319|
|  Spain|  Male|650.9920749279539|
|Germany|  Male|649.9665653495441|
|  Spain|Female|651.7695133149679|
+-------+------+-----------------+



[(('France', 'Female'), 649.1857585139319),
 (('Spain', 'Female'), 651.7695133149679),
 (('Spain', 'Male'), 650.9920749279539),
 (('France', 'Male'), 650.0646567381038),
 (('Germany', 'Female'), 653.0938809723386),
 (('Germany', 'Male'), 649.9665653495441)]

Crear una nueva columna “riesgo” basada en credit_score

Supongo que:
- credit_score >= 700 → bajo
- 600 <= credit_score < 700 → medio
- < 600 → alto


In [37]:
from pyspark.sql.functions import when

# Usando DataFrame
df_riesgo = df.withColumn("riesgo", 
    when(df.credit_score >= 700, "bajo")
    .when(df.credit_score >= 600, "medio")
    .otherwise("alto")
)
df_riesgo.select("customer_id", "credit_score", "riesgo").show()

# Usando RDD
# RDD original con columnas necesarias
rdd = df.select("customer_id", "credit_score").rdd
# Función para clasificar el riesgo
def clasificar_riesgo(row):
    score = row["credit_score"]
    if score >= 700:
        riesgo = "bajo"
    elif score >= 600:
        riesgo = "medio"
    else:
        riesgo = "alto"
    return (row["customer_id"], score, riesgo)
riesgo_rdd = rdd.map(clasificar_riesgo)
riesgo_rdd.collect()


+-----------+------------+------+
|customer_id|credit_score|riesgo|
+-----------+------------+------+
|   15634602|         619| medio|
|   15647311|         608| medio|
|   15619304|         502|  alto|
|   15701354|         699| medio|
|   15737888|         850|  bajo|
|   15574012|         645| medio|
|   15592531|         822|  bajo|
|   15656148|         376|  alto|
|   15792365|         501|  alto|
|   15592389|         684| medio|
|   15767821|         528|  alto|
|   15737173|         497|  alto|
|   15632264|         476|  alto|
|   15691483|         549|  alto|
|   15600882|         635| medio|
|   15643966|         616| medio|
|   15737452|         653| medio|
|   15788218|         549|  alto|
|   15661507|         587|  alto|
|   15568982|         726|  bajo|
+-----------+------------+------+
only showing top 20 rows


[(15634602, 619, 'medio'),
 (15647311, 608, 'medio'),
 (15619304, 502, 'alto'),
 (15701354, 699, 'medio'),
 (15737888, 850, 'bajo'),
 (15574012, 645, 'medio'),
 (15592531, 822, 'bajo'),
 (15656148, 376, 'alto'),
 (15792365, 501, 'alto'),
 (15592389, 684, 'medio'),
 (15767821, 528, 'alto'),
 (15737173, 497, 'alto'),
 (15632264, 476, 'alto'),
 (15691483, 549, 'alto'),
 (15600882, 635, 'medio'),
 (15643966, 616, 'medio'),
 (15737452, 653, 'medio'),
 (15788218, 549, 'alto'),
 (15661507, 587, 'alto'),
 (15568982, 726, 'bajo'),
 (15577657, 732, 'bajo'),
 (15597945, 636, 'medio'),
 (15699309, 510, 'alto'),
 (15725737, 669, 'medio'),
 (15625047, 846, 'bajo'),
 (15738191, 577, 'alto'),
 (15736816, 756, 'bajo'),
 (15700772, 571, 'alto'),
 (15728693, 574, 'alto'),
 (15656300, 411, 'alto'),
 (15589475, 591, 'alto'),
 (15706552, 533, 'alto'),
 (15750181, 553, 'alto'),
 (15659428, 520, 'alto'),
 (15732963, 722, 'bajo'),
 (15794171, 475, 'alto'),
 (15788448, 490, 'alto'),
 (15729599, 804, 'bajo'),
 (

### Simulación de procesamiento distribuido con RDD

Aquí se crea un RDD a partir de una lista de números y se divide en 4 particiones para simular el procesamiento paralelo. Se realizan operaciones básicas como suma y máximo para observar la paralelización.

In [38]:
# Crear un RDD con 4 particiones
numeros = list(range(1, 10001))
rdd_numeros = spark.sparkContext.parallelize(numeros,4)

# Verificar paralelización
print("Particiones:", rdd_numeros.getNumPartitions())

# Aplicar transformaciones
print("Suma:", rdd_numeros.sum())
print("Máximo:", rdd_numeros.max())

Particiones: 4
Suma: 50005000
Máximo: 10000


### 🧾 Conclusión

En este notebook se compararon transformaciones y agregaciones realizadas con **RDDs** y **DataFrames** en PySpark.

**Conclusiones principales:**

- **Los DataFrames** ofrecen una sintaxis más concisa, expresiva y optimizada, lo que facilita el desarrollo y mantenimiento del código.
- **Los RDDs**, aunque más flexibles y con mayor control de bajo nivel, requieren mayor esfuerzo y son más propensos a errores para tareas comunes como agrupaciones o cálculos agregados.
- Gracias a las optimizaciones del motor de ejecución de Spark (como **Catalyst** para planificación de consultas y **Tungsten** para ejecución optimizada), los DataFrames suelen ser **más eficientes** en rendimiento.
- Aun así, los RDDs pueden ser útiles en situaciones donde se trabaja con estructuras no tabulares o cuando se necesita un control más detallado sobre el flujo de datos.

✅ **Recomendación:** Para la mayoría de las tareas de análisis de datos en PySpark, se recomienda trabajar con **DataFrames** por su simplicidad y eficiencia. Los RDDs quedan reservados para casos específicos que justifiquen su uso.



#### Comparación entre RDD y DataFrame en PySpark

| Característica                | RDD                                   | DataFrame                             |
|------------------------------|-------------------------------------|-------------------------------------|
| Abstracción                  | Bajo nivel, colecciones distribuidas | Alto nivel, tablas similares a SQL  |
| Tipado                       | No estructurado, objetos genéricos  | Esquema definido con columnas       |
| Facilidad de uso             | Más complejo y verboso               | Más sencillo y declarativo           |
| Optimización                | Manual, sin optimizaciones automáticas | Optimización automática con Catalyst |
| Operaciones                 | Transformaciones con funciones lambda | Operaciones SQL y API expresiva      |
| Soporte para datos estructurados | Limitado                         | Excelente, con soporte para esquemas |
| Rendimiento                 | Generalmente más lento               | Generalmente más rápido               |
| Casos de uso recomendados    | Procesos muy personalizados, bajo control | Consultas, análisis exploratorio, ML |
